# Objective
Preprocess full KP55K NG data. 

In [ ]:
suppressPackageStartupMessages({
  library(here)
  library(haven)
  library(tidyverse)
  library(maplet)
})


**Preprocessing with maplet**

We perform the central NMR preprocessing steps using the `maplet` package. `maplet` is built around Bioconductor's SummarizedExperiment data structure, which integrates:

Patient metadata in colData

NMR biomarker concentrations in the assay matrix

Biomarker annotations in rowData, including biomarker names, pathway information, measurement units, and other feature-level metadata

For additional details on `maplet`, see the [publication](https://academic.oup.com/bioinformatics/article/38/4/1168/6409851?login=false) and [GitHub repository](https://github.com/krumsieklab/maplet).

In [ ]:
# load maplet object
D <- readRDS(here('preprocessing_inputs/kp55k_maplet.rds'))

dim(D) # print metabolites and sample dimensions

# Compute Missingness and Perform Preprocessing

In [ ]:
options(repr.plot.height = 5, repr.plot.width = 5)

# plot missingness heatmap of metabolites and samples 
D %>% mt_plots_missingness() %>% mtm_res_get_plots() %>% .[[2]]

We see here that there are a few metabolites with high missingness, and a few samples with really high missingness. We'll establish some filters for these missing features/samples here.

In [ ]:
# filter out high-missinngess metabolites and samples
D1 <- D %>% 
    # filter out metabolites with high-missingness 
    mt_pre_filter_missingness(feat_max = 0.5) %>% 
    # filter out samples with high missingess 
    mt_pre_filter_missingness(samp_max = 0.8)

This essentially removed `Glycerol` (>50% missingness) and 124 samples. 

## Imputation

Outliers were identified using a univariate approach with maplet::mt_pre_outlier_to_na(), with extreme values flagged based on the sample-size-adjusted threshold and replaced with NA prior to downstream preprocessing.

Since this NMR panel has a strong correlation structure, missing values (NA) were imputed using K-nearest neighbors (KNN), as correlated biomarker profiles make neighboring samples highly informative for estimating missing measurements without imposing a parametric distribution.

In [ ]:
# define maplet-compatabile KNN imputer
mt_pre_impute_knn_imputePkg <- function(
    D,
    k = 10,
    rowmax = 0.5,
    colmax = 0.8,
    maxp = 1500,
    rng.seed = 362436069
) {
    stopifnot("SummarizedExperiment" %in% class(D))
    if (!requireNamespace("impute", quietly = TRUE)) {
        stop("Package 'impute' is required for this function.")
    }
    X <- assay(D)
    # impute.knn expects features (metabolites) in rows, samples in columns
    imputed_X <- impute::impute.knn(
        as.matrix(X),
        k = k,
        rowmax = rowmax,
        colmax = colmax,
        maxp = maxp,
        rng.seed = rng.seed
    )$data
    assay(D) <- imputed_X

    funargs <- maplet:::mti_funargs()
    D <- maplet:::mti_generate_result(
        D,
        funargs = funargs,
        logtxt = sprintf("Imputed via impute.knn (k = %d)", k)
    )
    return(D)
}

In [ ]:
# reassemble maplet object and verify compatibility
D2 <- D1 %>% 
    # set outliers to NA
    mt_pre_outlier_to_na() %>% 
    # impute using new knn function
    mt_pre_impute_knn_imputePkg(k = 10)

D2 %>% mt_plots_missingness()

Values of zero, which indicate measurements below the limit of detection (LOD), were imputed using the minimum observed value for the corresponding biomarker.

In [ ]:
# set 0s to NA (these are below LOD)
D2 <- D2 %>% 
    mt_pre_zero_to_na() %>% 
    # impute these using min
    mt_pre_impute_min()

In [ ]:
# final exclusion criteria 
# all models rely on 'BASE' (age and sex - coded as M/F) - 
# we will  filter for participants with these values (omits 4 participants)
dim(D2)

# filter participants with M/F for BASE
D3 <- D2 %>% 
    mt_modify_filter_samples(filter = gender %in% c('M', 'F'))

dim(D3) # final metabolomics subcohort 

In [ ]:
# save maplet object
saveRDS(D3, here('preprocessing_inputs/kp55k_maplet_preprocessed.rds'))